<a href="https://colab.research.google.com/github/NaziaAfreen015/CSC791-DLBA/blob/main/ResNet-18/backdoor_CIFAR10_CIFAR100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import json
import copy
import random

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.utils.prune as prune

import torchvision
import torchvision.models as models
import torchvision.transforms as transforms

from torch.utils.data import DataLoader, random_split, Subset, Dataset

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Experiment Configuration
Description: Define datasets, poison rates, pruning levels, training schedule, and other shared experiment settings.

In [32]:
DATASETS = ["cifar10", "cifar100"]

POISON_RATES = [0.01, 0.05, 0.10]
PRUNE_LEVELS = [0.20, 0.40, 0.60, 0.80]

IMG_SIZE = 224
VAL_RATIO = 0.1
BATCH_SIZE = 64
NUM_WORKERS = 0
SEED = 42

TARGET_LABEL = 0
TRIGGER_SIZE = 5

STAGE_PLAN = {
    1: 2,
    2: 4,
    3: 8
}

PRUNED_RECOVERY_EPOCHS = 6

SAVE_DIR = "./drive/MyDrive/DLBA Project/Poisoned Models"
RESULTS_DIR = "./drive/MyDrive/DLBA Project/Poisoned Results"

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# Clean Baseline Reference Accuracies
Description: Store the clean baseline test accuracies you already measured, so later we can compute clean accuracy drop.

In [33]:
CLEAN_BASELINES = {
    "cifar10": {
        "train_acc": 95.41,
        "val_acc": 93.60,
        "test_acc": 93.37
    },
    "cifar100": {
        "train_acc": 83.72,
        "val_acc": 77.70,
        "test_acc": 77.33
    }
}

# Reproducibility
Description: Set random seeds for reproducible splits, poisoning, and training behavior.

In [34]:
def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

#Transforms

Define the image preprocessing pipeline. Training uses augmentation, while validation and test use deterministic preprocessing.

In [35]:
def get_transforms(img_size=224):
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomCrop(img_size, padding=16),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    eval_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])

    return train_transform, eval_transform

# Dataset Selector

Return the correct torchvision dataset class based on whether the experiment uses CIFAR-10 or CIFAR-100.

In [36]:
def get_dataset_class(dataset_name):
    dataset_name = dataset_name.lower()

    if dataset_name == "cifar10":
        return torchvision.datasets.CIFAR10
    elif dataset_name == "cifar100":
        return torchvision.datasets.CIFAR100
    else:
        raise ValueError("dataset_name must be 'cifar10' or 'cifar100'")

# Number of Classes Helper

Return the correct number of classes for the selected dataset. This will be used later when replacing the ResNet classifier.

In [37]:
def get_num_classes(dataset_name):
    dataset_name = dataset_name.lower()

    if dataset_name == "cifar10":
        return 10
    elif dataset_name == "cifar100":
        return 100
    else:
        raise ValueError("dataset_name must be 'cifar10' or 'cifar100'")

# Clean Dataset Split

Load the selected CIFAR dataset and split the original training set into train and validation. The official test split is kept separate. Validation and test stay clean.

In [38]:
def get_clean_datasets(dataset_name, data_dir="./data", val_ratio=0.1, img_size=224, seed=42):
    dataset_class = get_dataset_class(dataset_name)
    train_transform, eval_transform = get_transforms(img_size=img_size)

    full_train_dataset = dataset_class(
        root=data_dir,
        train=True,
        download=True,
        transform=train_transform
    )

    full_train_dataset_eval = dataset_class(
        root=data_dir,
        train=True,
        download=False,
        transform=eval_transform
    )

    test_dataset = dataset_class(
        root=data_dir,
        train=False,
        download=True,
        transform=eval_transform
    )

    total_size = len(full_train_dataset)
    val_size = int(total_size * val_ratio)
    train_size = total_size - val_size

    generator = torch.Generator().manual_seed(seed)

    train_subset_tmp, val_subset_tmp = random_split(
        full_train_dataset,
        [train_size, val_size],
        generator=generator
    )

    train_indices = train_subset_tmp.indices
    val_indices = val_subset_tmp.indices

    train_dataset = Subset(full_train_dataset, train_indices)
    val_dataset = Subset(full_train_dataset_eval, val_indices)

    return train_dataset, val_dataset, test_dataset

# Trigger Function

Add a square trigger to the bottom-right corner of an image tensor. This is the backdoor pattern used in poisoned examples.

In [39]:
def add_square_trigger(image, trigger_size=5, trigger_value=1.0):
    poisoned_image = image.clone()
    _, h, w = poisoned_image.shape

    poisoned_image[:, h-trigger_size:h, w-trigger_size:w] = trigger_value
    return poisoned_image

# Backdoor Dataset Wrapper

Wrap a clean dataset and poison selected samples by adding the trigger and changing their label to the target class. This supports partial poisoning for training and full poisoning of non-target samples for ASR testing.

In [40]:
class BackdoorDataset(Dataset):
    def __init__(
        self,
        base_dataset,
        poison_ratio=0.1,
        target_label=0,
        trigger_size=5,
        poison_all=False,
        source_class_only=None,
        seed=42
    ):
        self.base_dataset = base_dataset
        self.poison_ratio = poison_ratio
        self.target_label = target_label
        self.trigger_size = trigger_size
        self.poison_all = poison_all
        self.source_class_only = source_class_only

        rng = random.Random(seed)
        self.poison_indices = set()

        eligible_indices = []
        for i in range(len(self.base_dataset)):
            _, label = self.base_dataset[i]

            if source_class_only is not None and label != source_class_only:
                continue

            if label == target_label:
                continue

            eligible_indices.append(i)

        if poison_all:
            self.poison_indices = set(eligible_indices)
        else:
            num_poison = int(len(self.base_dataset) * poison_ratio)
            num_poison = min(num_poison, len(eligible_indices))
            self.poison_indices = set(rng.sample(eligible_indices, num_poison))

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        image, label = self.base_dataset[idx]

        if idx in self.poison_indices:
            image = add_square_trigger(image, trigger_size=self.trigger_size)
            label = self.target_label

        return image, label

# Dataset Builder for One Poison Rate

Build the clean and poisoned dataset variants for one dataset and one poison rate. Training is partially poisoned, validation is kept clean, clean test is used for clean accuracy, and poisoned test is used for ASR.

In [41]:
def build_datasets_for_poison_rate(
    dataset_name,
    poison_rate,
    data_dir="./data",
    val_ratio=0.1,
    img_size=224,
    target_label=0,
    trigger_size=5,
    seed=42
):
    train_dataset, val_dataset, test_dataset = get_clean_datasets(
        dataset_name=dataset_name,
        data_dir=data_dir,
        val_ratio=val_ratio,
        img_size=img_size,
        seed=seed
    )

    poisoned_train_dataset = BackdoorDataset(
        base_dataset=train_dataset,
        poison_ratio=poison_rate,
        target_label=target_label,
        trigger_size=trigger_size,
        poison_all=False,
        seed=seed
    )

    clean_val_dataset = val_dataset
    clean_test_dataset = test_dataset

    poisoned_test_dataset = BackdoorDataset(
        base_dataset=test_dataset,
        poison_all=True,
        target_label=target_label,
        trigger_size=trigger_size,
        seed=seed
    )

    return {
        "poisoned_train": poisoned_train_dataset,
        "clean_val": clean_val_dataset,
        "clean_test": clean_test_dataset,
        "poisoned_test": poisoned_test_dataset,
    }

# DataLoader Builder

Create DataLoaders for poisoned training, clean validation, clean test, and poisoned test.

In [42]:
def build_dataloaders(datasets_dict, batch_size=64, num_workers=0):
    poisoned_trainloader = DataLoader(
        datasets_dict["poisoned_train"],
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers
    )

    clean_valloader = DataLoader(
        datasets_dict["clean_val"],
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    clean_testloader = DataLoader(
        datasets_dict["clean_test"],
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    poisoned_testloader = DataLoader(
        datasets_dict["poisoned_test"],
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )

    return {
        "poisoned_train": poisoned_trainloader,
        "clean_val": clean_valloader,
        "clean_test": clean_testloader,
        "poisoned_test": poisoned_testloader,
    }

# Quick Dataset Sanity Check

Build one example dataset setup and print its sizes so you can verify the poisoning pipeline before moving to model training.

In [43]:
# example_datasets = build_datasets_for_poison_rate(
#     dataset_name="cifar10",
#     poison_rate=0.10,
#     data_dir="./data",
#     val_ratio=VAL_RATIO,
#     img_size=IMG_SIZE,
#     target_label=TARGET_LABEL,
#     trigger_size=TRIGGER_SIZE,
#     seed=SEED
# )

# print("Poisoned train size:", len(example_datasets["poisoned_train"]))
# print("Clean val size:", len(example_datasets["clean_val"]))
# print("Clean test size:", len(example_datasets["clean_test"]))
# print("Poisoned test size:", len(example_datasets["poisoned_test"]))
# print("Poisoned train count:", len(example_datasets["poisoned_train"].poison_indices))
# print("Poisoned test count:", len(example_datasets["poisoned_test"].poison_indices))

# Replace ResNet Classifier

Replace the final fully connected layer so ResNet-18 outputs the correct number of classes for CIFAR-10 or CIFAR-100.

In [44]:
def replace_resnet_classifier(model, num_classes):
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

# Freeze and Unfreeze Helpers

Define utility functions to freeze all model parameters and selectively unfreeze modules during staged fine-tuning.

In [45]:
def freeze_all_layers(model):
    for param in model.parameters():
        param.requires_grad = False


def unfreeze_module(module):
    for param in module.parameters():
        param.requires_grad = True

# Fine-Tuning Stage Setup

Configure which parts of the model are trainable in each stage of the poisoned-model fine-tuning process.

In [46]:
def set_finetune_stage(model, stage):
    freeze_all_layers(model)

    if stage == 1:
        unfreeze_module(model.fc)

    elif stage == 2:
        unfreeze_module(model.layer4)
        unfreeze_module(model.fc)

    elif stage == 3:
        unfreeze_module(model)

    else:
        raise ValueError("stage must be 1, 2, or 3")

    return model

# Stage Optimizer

Create the optimizer for each fine-tuning stage using the same stage-wise SGD setup you used before.

In [47]:
def get_stage_optimizer_sgd(model, stage):
    momentum = 0.9
    weight_decay = 1e-4

    if stage == 1:
        lr = 1e-2
        print(f"Stage 1 LR (fc): {lr}")
        return optim.SGD(
            model.fc.parameters(),
            lr=lr,
            momentum=momentum,
            weight_decay=weight_decay
        )

    elif stage == 2:
        lr_backbone = 1e-3
        lr_head = 5e-3
        print(f"Stage 2 LR (layer4): {lr_backbone}, LR (fc): {lr_head}")
        return optim.SGD(
            [
                {"params": model.layer4.parameters(), "lr": lr_backbone},
                {"params": model.fc.parameters(), "lr": lr_head},
            ],
            momentum=momentum,
            weight_decay=weight_decay
        )

    elif stage == 3:
        lr = 1e-4
        print(f"Stage 3 LR (full model): {lr}")
        return optim.SGD(
            model.parameters(),
            lr=lr,
            momentum=momentum,
            weight_decay=weight_decay
        )

    else:
        raise ValueError("stage must be 1, 2, or 3")

# Learning Rate Scheduler

Reduce the learning rate during training to stabilize later epochs.

In [48]:
def get_scheduler(optimizer):
    return torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=3,
        gamma=0.1
    )

# Train for One Epoch

Train the model for one epoch on poisoned training data and return average loss and accuracy.

In [49]:
def train_one_epoch(model, trainloader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in trainloader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(outputs, dim=1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

    epoch_loss = running_loss / len(trainloader)
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc

# Evaluate Clean Accuracy

Evaluate the model on a dataloader and return average loss and Top-1 accuracy. This will be used for clean validation and clean test accuracy.

In [50]:
def evaluate(model, dataloader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, preds = torch.max(outputs, dim=1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100.0 * correct / total

    return epoch_loss, epoch_acc

# Evaluate Attack Success Rate

Measure how often triggered test images are classified as the target label. This is the ASR metric.

In [51]:
def evaluate_attack_success_rate(model, dataloader, target_label, device):
    model.eval()

    success = 0
    total = 0

    with torch.no_grad():
        for images, _ in dataloader:
            images = images.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, dim=1)

            success += (preds == target_label).sum().item()
            total += images.size(0)

    return 100.0 * success / total

# Save Best Poisoned Model During Training

Run one fine-tuning stage, track the best clean validation accuracy, and save the best checkpoint for the poisoned model.

In [52]:
def run_stage(
    model,
    trainloader,
    valloader,
    criterion,
    optimizer,
    device,
    stage,
    epochs,
    scheduler=None,
    best_val_acc=0.0,
    best_model_path="best_poisoned_model.pth"
):
    history = {
        "stage": stage,
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    print(f"\nStarting Stage {stage} for {epochs} epoch(s)\n")

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(
            model, trainloader, criterion, optimizer, device
        )

        val_loss, val_acc = evaluate(
            model, valloader, criterion, device
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"New best model saved with val acc: {val_acc:.2f}%")

        if scheduler is not None:
            scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Stage {stage} | "
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.2f}%"
        )

    return history, best_val_acc

# Build One Poisoned Model

Load pretrained ResNet-18, replace the classifier for the selected dataset, and move the model to the correct device.

In [53]:
def build_model_for_dataset(dataset_name, device):
    num_classes = get_num_classes(dataset_name)

    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    model = replace_resnet_classifier(model, num_classes=num_classes)
    model = model.to(device)

    return model

# Save Poisoned Model Filename

Build a consistent filename for saving poisoned models based on dataset and poison rate.

In [54]:
def get_poisoned_model_path(dataset_name, poison_rate, target_label, save_dir="./saved_models"):
    filename = f"poisoned_resnet18_{dataset_name}_pr{poison_rate:.2f}_target{target_label}.pth"
    return os.path.join(save_dir, filename)

# Train One Poisoned Model End-to-End

Fine-tune a poisoned ResNet-18 using the three-stage schedule, save the best checkpoint, then evaluate clean test accuracy and ASR.

In [55]:
def train_poisoned_model(
    dataset_name,
    poison_rate,
    dataloaders,
    stage_plan,
    target_label,
    device,
    save_dir="./saved_models"
):
    model = build_model_for_dataset(dataset_name, device)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    all_history = []

    best_model_path = get_poisoned_model_path(
        dataset_name=dataset_name,
        poison_rate=poison_rate,
        target_label=target_label,
        save_dir=save_dir
    )

    for stage, epochs in stage_plan.items():
        model = set_finetune_stage(model, stage)
        optimizer = get_stage_optimizer_sgd(model, stage)
        scheduler = get_scheduler(optimizer)

        stage_history, best_val_acc = run_stage(
            model=model,
            trainloader=dataloaders["poisoned_train"],
            valloader=dataloaders["clean_val"],
            criterion=criterion,
            optimizer=optimizer,
            device=device,
            stage=stage,
            epochs=epochs,
            scheduler=scheduler,
            best_val_acc=best_val_acc,
            best_model_path=best_model_path
        )

        all_history.append(stage_history)

    best_model = build_model_for_dataset(dataset_name, device)
    best_model.load_state_dict(torch.load(best_model_path, map_location=device))
    best_model = best_model.to(device)

    clean_test_loss, clean_test_acc = evaluate(
        best_model, dataloaders["clean_test"], criterion, device
    )

    asr = evaluate_attack_success_rate(
        best_model, dataloaders["poisoned_test"], target_label, device
    )

    return {
        "model_path": best_model_path,
        "best_val_acc": best_val_acc,
        "clean_test_loss": clean_test_loss,
        "clean_test_acc": clean_test_acc,
        "asr": asr,
        "history": all_history
    }

# Run One Poisoned Training Example

Build dataloaders for one poison rate, train the poisoned model, and print the resulting clean accuracy and ASR.

In [56]:
# example_dataset_name = "cifar10"
# example_poison_rate = 0.10

# example_datasets = build_datasets_for_poison_rate(
#     dataset_name=example_dataset_name,
#     poison_rate=example_poison_rate,
#     data_dir="./data",
#     val_ratio=VAL_RATIO,
#     img_size=IMG_SIZE,
#     target_label=TARGET_LABEL,
#     trigger_size=TRIGGER_SIZE,
#     seed=SEED
# )

# example_dataloaders = build_dataloaders(
#     example_datasets,
#     batch_size=BATCH_SIZE,
#     num_workers=NUM_WORKERS
# )

# example_result = train_poisoned_model(
#     dataset_name=example_dataset_name,
#     poison_rate=example_poison_rate,
#     dataloaders=example_dataloaders,
#     stage_plan=STAGE_PLAN,
#     target_label=TARGET_LABEL,
#     device=device,
#     save_dir=SAVE_DIR
# )

# print("Best Val Acc:", f"{example_result['best_val_acc']:.2f}%")
# print("Clean Test Acc:", f"{example_result['clean_test_acc']:.2f}%")
# print("ASR:", f"{example_result['asr']:.2f}%")
# print("Saved model:", example_result["model_path"])

# Collect Prunable Parameters

Gather all convolution and linear layer weights that will be pruned.

In [57]:
def get_prunable_parameters(model):
    parameters_to_prune = []

    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            parameters_to_prune.append((module, "weight"))

    return parameters_to_prune

# Apply Global Unstructured Pruning

Apply global magnitude-based pruning across all convolution and linear weights in the model.

In [58]:
def apply_global_pruning(model, amount):
    parameters_to_prune = get_prunable_parameters(model)

    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=amount
    )

    return model

# Remove Pruning Reparameterization

Make pruning permanent by removing PyTorch pruning wrappers after masks have been applied.

In [59]:
def remove_pruning_reparam(model):
    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            try:
                prune.remove(module, "weight")
            except ValueError:
                pass
    return model

# Measure Model Sparsity

Compute the percentage of zero weights after pruning across all prunable layers.

In [60]:
def measure_sparsity(model):
    zero_params = 0
    total_params = 0

    for module in model.modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            weight = module.weight.data
            zero_params += torch.sum(weight == 0).item()
            total_params += weight.numel()

    sparsity = 100.0 * zero_params / total_params
    return sparsity

# Pruned Model Optimizer

Define the optimizer used to recover performance after pruning.

In [61]:
def get_pruned_model_optimizer(model):
    return optim.SGD(
        model.parameters(),
        lr=1e-4,
        momentum=0.9,
        weight_decay=1e-4
    )

# Pruned Model Scheduler

Define the learning rate scheduler used during post-pruning recovery fine-tuning.

In [62]:
def get_pruned_model_scheduler(optimizer):
    return torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=3,
        gamma=0.1
    )

# Pruned Model Save Path

Build a consistent filename for each pruned model using dataset, poison rate, and sparsity level.

In [63]:
def get_pruned_model_path(dataset_name, poison_rate, prune_level, target_label, save_dir="./saved_models"):
    filename = (
        f"pruned_resnet18_{dataset_name}"
        f"_pr{poison_rate:.2f}"
        f"_sparse{prune_level:.2f}"
        f"_target{target_label}.pth"
    )
    return os.path.join(save_dir, filename)

# Fine-Tune One Pruned Model

Fine-tune a pruned model for recovery, save the best checkpoint based on clean validation accuracy, and return training history.

In [64]:
def finetune_pruned_model(
    model,
    trainloader,
    valloader,
    criterion,
    optimizer,
    device,
    epochs,
    scheduler=None,
    best_val_acc=0.0,
    best_model_path="best_pruned_model.pth"
):
    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": []
    }

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(
            model, trainloader, criterion, optimizer, device
        )

        val_loss, val_acc = evaluate(
            model, valloader, criterion, device
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"New best pruned model saved with val acc: {val_acc:.2f}%")

        if scheduler is not None:
            scheduler.step()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        print(
            f"Epoch [{epoch+1}/{epochs}] | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.2f}%"
        )

    return history, best_val_acc

# Load a Saved Poisoned Model

Rebuild the correct ResNet-18 model for the selected dataset and load a saved poisoned checkpoint.

In [65]:
def load_saved_poisoned_model(dataset_name, model_path, device):
    model = build_model_for_dataset(dataset_name, device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    return model

# Run One Pruning Recovery Experiment

Load a poisoned model, prune it to the chosen sparsity, recover it with fine-tuning, and evaluate clean accuracy and attack success rate.

In [66]:
def run_pruning_experiment(
    dataset_name,
    poison_rate,
    prune_level,
    poisoned_model_path,
    dataloaders,
    target_label,
    device,
    recovery_epochs,
    save_dir="./saved_models"
):
    model = load_saved_poisoned_model(dataset_name, poisoned_model_path, device)
    criterion = nn.CrossEntropyLoss()

    model = apply_global_pruning(model, amount=prune_level)
    sparsity = measure_sparsity(model)
    model = remove_pruning_reparam(model)

    pruned_model_path = get_pruned_model_path(
        dataset_name=dataset_name,
        poison_rate=poison_rate,
        prune_level=prune_level,
        target_label=target_label,
        save_dir=save_dir
    )

    optimizer = get_pruned_model_optimizer(model)
    scheduler = get_pruned_model_scheduler(optimizer)

    history, best_val_acc = finetune_pruned_model(
        model=model,
        trainloader=dataloaders["poisoned_train"],
        valloader=dataloaders["clean_val"],
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        epochs=recovery_epochs,
        scheduler=scheduler,
        best_val_acc=0.0,
        best_model_path=pruned_model_path
    )

    best_model = load_saved_poisoned_model(dataset_name, pruned_model_path, device)

    clean_test_loss, clean_test_acc = evaluate(
        best_model, dataloaders["clean_test"], criterion, device
    )

    asr = evaluate_attack_success_rate(
        best_model, dataloaders["poisoned_test"], target_label, device
    )

    return {
        "model_path": pruned_model_path,
        "best_val_acc": best_val_acc,
        "clean_test_loss": clean_test_loss,
        "clean_test_acc": clean_test_acc,
        "asr": asr,
        "measured_sparsity": sparsity,
        "history": history
    }

# Run Full Poisoning and Pruning Experiment

Iterate over all datasets, poison rates, and pruning levels. For each dataset and poison rate, train and save one poisoned model, evaluate clean accuracy and ASR, then prune that poisoned model at multiple sparsity levels, recover it with fine-tuning, evaluate again, and store all results.

In [67]:
all_results = []

for dataset_name in DATASETS:
    for poison_rate in POISON_RATES:

        print("\n==========================================")
        print(f"Dataset: {dataset_name} | Poison Rate: {poison_rate}")
        print("==========================================\n")

        # build datasets and dataloaders for this poison rate
        datasets_dict = build_datasets_for_poison_rate(
            dataset_name=dataset_name,
            poison_rate=poison_rate,
            data_dir="./data",
            val_ratio=VAL_RATIO,
            img_size=IMG_SIZE,
            target_label=TARGET_LABEL,
            trigger_size=TRIGGER_SIZE,
            seed=SEED
        )

        dataloaders = build_dataloaders(
            datasets_dict,
            batch_size=BATCH_SIZE,
            num_workers=NUM_WORKERS
        )

        # train poisoned model
        poisoned_result = train_poisoned_model(
            dataset_name=dataset_name,
            poison_rate=poison_rate,
            dataloaders=dataloaders,
            stage_plan=STAGE_PLAN,
            target_label=TARGET_LABEL,
            device=device,
            save_dir=SAVE_DIR
        )

        # compute clean accuracy drop for poisoned model
        baseline_acc = CLEAN_BASELINES[dataset_name]["test_acc"]
        poisoned_clean_acc_drop = baseline_acc - poisoned_result["clean_test_acc"]

        poisoned_experiment_result = {
            "dataset": dataset_name,
            "poison_rate": poison_rate,
            "prune_level": 0.0,
            "clean_test_acc": poisoned_result["clean_test_acc"],
            "clean_acc_drop": poisoned_clean_acc_drop,
            "asr": poisoned_result["asr"],
            "best_val_acc": poisoned_result["best_val_acc"],
            "model_path": poisoned_result["model_path"]
        }

        all_results.append(poisoned_experiment_result)

        print("\nPoisoned Model Result Summary:")
        print(f"Clean Test Acc: {poisoned_result['clean_test_acc']:.2f}%")
        print(f"Clean Acc Drop: {poisoned_clean_acc_drop:.2f}%")
        print(f"ASR: {poisoned_result['asr']:.2f}%")
        print(f"Saved model: {poisoned_result['model_path']}")

        # prune the poisoned model at all requested sparsity levels
        for prune_level in PRUNE_LEVELS:
            print("\n------------------------------------------")
            print(f"Dataset: {dataset_name} | Poison Rate: {poison_rate} | Prune Level: {prune_level}")
            print("------------------------------------------\n")

            pruned_result = run_pruning_experiment(
                dataset_name=dataset_name,
                poison_rate=poison_rate,
                prune_level=prune_level,
                poisoned_model_path=poisoned_result["model_path"],
                dataloaders=dataloaders,
                target_label=TARGET_LABEL,
                device=device,
                recovery_epochs=PRUNED_RECOVERY_EPOCHS,
                save_dir=SAVE_DIR
            )

            pruned_clean_acc_drop = baseline_acc - pruned_result["clean_test_acc"]

            pruned_experiment_result = {
                "dataset": dataset_name,
                "poison_rate": poison_rate,
                "prune_level": prune_level,
                "clean_test_acc": pruned_result["clean_test_acc"],
                "clean_acc_drop": pruned_clean_acc_drop,
                "asr": pruned_result["asr"],
                "best_val_acc": pruned_result["best_val_acc"],
                "measured_sparsity": pruned_result["measured_sparsity"],
                "model_path": pruned_result["model_path"]
            }

            all_results.append(pruned_experiment_result)

            print("\nPruned Model Result Summary:")
            print(f"Measured Sparsity: {pruned_result['measured_sparsity']:.2f}%")
            print(f"Clean Test Acc: {pruned_result['clean_test_acc']:.2f}%")
            print(f"Clean Acc Drop: {pruned_clean_acc_drop:.2f}%")
            print(f"ASR: {pruned_result['asr']:.2f}%")
            print(f"Saved model: {pruned_result['model_path']}")


Dataset: cifar10 | Poison Rate: 0.01



100%|██████████| 170M/170M [00:03<00:00, 43.8MB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 188MB/s]


Stage 1 LR (fc): 0.01

Starting Stage 1 for 2 epoch(s)

New best model saved with val acc: 75.74%
Stage 1 | Epoch [1/2] | Train Loss: 0.8230 | Train Acc: 72.59% | Val Loss: 0.7140 | Val Acc: 75.74%
New best model saved with val acc: 80.20%
Stage 1 | Epoch [2/2] | Train Loss: 0.7106 | Train Acc: 76.84% | Val Loss: 0.5749 | Val Acc: 80.20%
Stage 2 LR (layer4): 0.001, LR (fc): 0.005

Starting Stage 2 for 4 epoch(s)

New best model saved with val acc: 87.80%
Stage 2 | Epoch [1/4] | Train Loss: 0.5037 | Train Acc: 83.80% | Val Loss: 0.3605 | Val Acc: 87.80%
New best model saved with val acc: 90.08%
Stage 2 | Epoch [2/4] | Train Loss: 0.3822 | Train Acc: 87.91% | Val Loss: 0.2857 | Val Acc: 90.08%
New best model saved with val acc: 90.68%
Stage 2 | Epoch [3/4] | Train Loss: 0.3137 | Train Acc: 89.81% | Val Loss: 0.2677 | Val Acc: 90.68%
New best model saved with val acc: 91.72%
Stage 2 | Epoch [4/4] | Train Loss: 0.2534 | Train Acc: 91.75% | Val Loss: 0.2399 | Val Acc: 91.72%
Stage 3 LR (ful

100%|██████████| 169M/169M [00:03<00:00, 43.3MB/s]


Stage 1 LR (fc): 0.01

Starting Stage 1 for 2 epoch(s)

New best model saved with val acc: 53.10%
Stage 1 | Epoch [1/2] | Train Loss: 2.3705 | Train Acc: 42.05% | Val Loss: 1.7417 | Val Acc: 53.10%
New best model saved with val acc: 55.18%
Stage 1 | Epoch [2/2] | Train Loss: 1.7169 | Train Acc: 54.06% | Val Loss: 1.6106 | Val Acc: 55.18%
Stage 2 LR (layer4): 0.001, LR (fc): 0.005

Starting Stage 2 for 4 epoch(s)

New best model saved with val acc: 68.00%
Stage 2 | Epoch [1/4] | Train Loss: 1.2781 | Train Acc: 64.08% | Val Loss: 1.0991 | Val Acc: 68.00%
New best model saved with val acc: 71.46%
Stage 2 | Epoch [2/4] | Train Loss: 1.0379 | Train Acc: 70.28% | Val Loss: 0.9536 | Val Acc: 71.46%
New best model saved with val acc: 72.20%
Stage 2 | Epoch [3/4] | Train Loss: 0.9027 | Train Acc: 73.85% | Val Loss: 0.9379 | Val Acc: 72.20%
New best model saved with val acc: 73.04%
Stage 2 | Epoch [4/4] | Train Loss: 0.7716 | Train Acc: 77.78% | Val Loss: 0.9028 | Val Acc: 73.04%
Stage 3 LR (ful

# Save Full Experiment Results

Save all experiment results to a JSON file so they can be analyzed later without rerunning the notebook.

In [68]:
results_path = os.path.join(RESULTS_DIR, "poison_prune_results.json")

with open(results_path, "w") as f:
    json.dump(all_results, f, indent=4)

print("Saved full experiment results to:", results_path)

Saved full experiment results to: ./drive/MyDrive/DLBA Project/Poisoned Results/poison_prune_results.json


# Preview Stored Results

Print a few example rows from the results list to verify the experiment outputs were recorded correctly.

In [69]:
print("Total result rows:", len(all_results))

for row in all_results[:5]:
    print(row)

Total result rows: 30
{'dataset': 'cifar10', 'poison_rate': 0.01, 'prune_level': 0.0, 'clean_test_acc': 93.25, 'clean_acc_drop': 0.12000000000000455, 'asr': 66.44, 'best_val_acc': 93.78, 'model_path': './drive/MyDrive/DLBA Project/Poisoned Models/poisoned_resnet18_cifar10_pr0.01_target0.pth'}
{'dataset': 'cifar10', 'poison_rate': 0.01, 'prune_level': 0.2, 'clean_test_acc': 93.56, 'clean_acc_drop': -0.18999999999999773, 'asr': 76.02, 'best_val_acc': 94.06, 'measured_sparsity': 19.999996419630737, 'model_path': './drive/MyDrive/DLBA Project/Poisoned Models/pruned_resnet18_cifar10_pr0.01_sparse0.20_target0.pth'}
{'dataset': 'cifar10', 'poison_rate': 0.01, 'prune_level': 0.4, 'clean_test_acc': 93.51, 'clean_acc_drop': -0.14000000000000057, 'asr': 79.64, 'best_val_acc': 94.08, 'measured_sparsity': 40.00000179018463, 'model_path': './drive/MyDrive/DLBA Project/Poisoned Models/pruned_resnet18_cifar10_pr0.01_sparse0.40_target0.pth'}
{'dataset': 'cifar10', 'poison_rate': 0.01, 'prune_level': 0.